# Two-way MQTT chat with MongoDB and FastAPI
Run the numbered cells in order. Alice and Bob each publish on their own topic and subscribe to the other person’s topic. The API and both clients share one MongoDB message history.

## 1. Start the services
Use the `IOT` project folder containing `mqtt_chat.py` and the updated `api.py`. Keep your existing Mosquitto broker running on port 1883. If needed, start it in a terminal with `/opt/homebrew/sbin/mosquitto -p 1883`. Start the existing MongoDB container with `docker start iot-mongodb`. In another terminal run:

```bash
cd /Users/htet/Desktop/IOT/A1/IOT
.venv/bin/python -m uvicorn api:app --reload
```
If the API is already running from this folder, reuse it. Open this notebook in VS Code or Jupyter and choose a Python 3 kernel.

## 2. Check the API and its connections

In [ ]:
import json
from urllib.request import Request, urlopen

BASE = "http://127.0.0.1:8000"

def api(path, body=None):
    data = None if body is None else json.dumps(body).encode()
    request = Request(BASE + path, data=data, headers={"Content-Type": "application/json"})
    with urlopen(request, timeout=10) as response:
        return json.load(response)

health = api("/health")
print(health)
assert health["status"] == "healthy", "Check MongoDB and Mosquitto."
assert "/chat/messages" in api("/openapi.json")["paths"], "Restart the updated API from IOT."

## 3. Connect Alice and Bob
Open two new Terminal tabs. Run one command per tab and wait for both to print **Connected**.

**Alice**
```bash
cd /Users/htet/Desktop/IOT/A1/IOT && .venv/bin/python mqtt_chat.py alice
```

**Bob**
```bash
cd /Users/htet/Desktop/IOT/A1/IOT && .venv/bin/python mqtt_chat.py bob
```
Alice publishes on `iot/demo/chat/alice`, which Bob subscribes to. Bob publishes on `iot/demo/chat/bob`, which Alice subscribes to.

## 4. Try both directions
In Alice’s terminal type `Hello Bob` and press Enter. Look for `alice: Hello Bob` in Bob’s terminal. Reply `Hello Alice` from Bob and check Alice’s terminal. Each sender saves the message before publishing.

## 5. Send a message through REST
Run this cell once and watch Bob’s terminal. Each run creates a new message. A broker acknowledgment confirms Mosquitto received the publish.

In [ ]:
sent = api("/chat/messages", {"sender": "alice", "text": "Hello Bob from the notebook"})
print(sent)
print("Broker acknowledged:", sent["broker_acknowledged"])

## 6. Read MongoDB history through the API
Compare the output with `/history` in either terminal. The API returns the most recent messages in chronological order.

In [ ]:
history = api("/chat/messages?limit=10")
for message in history:
    print(message["timestamp"], message["sender"], "→", message["recipient"], message["text"])
print("Notebook message found:", any(m["_id"] == sent["message"]["_id"] for m in history))

## 7. Check persistence and finish
Type `/quit` in Bob’s terminal. Send another message from Alice, restart Bob with the same command, and type `/history`. Bob can read the stored message after reconnecting. Type `/quit` in each client when finished.

**Troubleshooting**  
Connection refused on port 8000 means the API is unreachable. A 404 for `/chat/messages` means the running API lacks the chat route. If history works but live messages are missing, check that both terminals say **Connected** and use different names.